# Week 6 Decision Trees & Random Forests

**Datasets:** UCL Online Retail II · Olist Brazilian E-Commerce

## Topics
1. **Decision tree (UCL)** — classify top-quartile spenders from recency, log frequency, basket size, and item count.
2. **Random forest (UCL + Olist)** — compare a single tree to an ensemble and tune depth and leaf size on both datasets.


## Data loading and cleaning

Load UCL customer RFM and Olist order-level data.


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

np.random.seed(42)
DATA_DIR = Path(".")

orders = pd.read_csv(DATA_DIR / "olist_Brazilian" / "olist_orders_dataset.csv")
reviews = pd.read_csv(DATA_DIR / "olist_Brazilian" / "olist_order_reviews_dataset.csv")
items = pd.read_csv(DATA_DIR / "olist_Brazilian" / "olist_order_items_dataset.csv")
payments = pd.read_csv(DATA_DIR / "olist_Brazilian" / "olist_order_payments_dataset.csv")
customers = pd.read_csv(DATA_DIR / "olist_Brazilian" / "olist_customers_dataset.csv")

pay = payments.groupby("order_id").agg(
    payment_value=("payment_value", "sum"),
    payment_installments=("payment_installments", "max"),
    n_payments=("payment_sequential", "max"),
).reset_index()
itm = items.groupby("order_id").agg(
    total_price=("price", "sum"),
    freight_value=("freight_value", "sum"),
    n_items=("order_item_id", "count"),
).reset_index()
rev = reviews.groupby("order_id").agg(review_score=("review_score", "mean")).reset_index()

olist_df = (
    orders.merge(rev, on="order_id")
    .merge(itm, on="order_id")
    .merge(pay, on="order_id")
    .merge(customers[["customer_id", "customer_unique_id", "customer_state"]], on="customer_id")
)
olist_df["order_purchase_timestamp"] = pd.to_datetime(olist_df["order_purchase_timestamp"])
olist_df["order_month"] = olist_df["order_purchase_timestamp"].dt.month
olist_df["delivery_days"] = (
    pd.to_datetime(olist_df["order_delivered_customer_date"], errors="coerce")
    - olist_df["order_purchase_timestamp"]
).dt.days
olist_df["delivery_days"] = olist_df["delivery_days"].fillna(olist_df["delivery_days"].median())
olist_df = olist_df.dropna(subset=["review_score", "payment_value", "total_price"])

ucl_raw = pd.read_excel(DATA_DIR / "ucl_dataset.xlsx")
ucl_raw = ucl_raw.rename(columns={"Invoice": "InvoiceNo", "Price": "UnitPrice", "Customer ID": "CustomerID"})
ucl_raw["InvoiceDate"] = pd.to_datetime(ucl_raw["InvoiceDate"])
ucl_raw = ucl_raw.dropna(subset=["CustomerID"])
ucl_raw = ucl_raw[~ucl_raw["InvoiceNo"].astype(str).str.startswith("C", na=False)]
ucl_raw = ucl_raw[(ucl_raw["Quantity"] > 0) & (ucl_raw["UnitPrice"] > 0)]
ucl_raw["LineTotal"] = ucl_raw["Quantity"] * ucl_raw["UnitPrice"]
anchor = ucl_raw["InvoiceDate"].max() + pd.Timedelta(days=1)
ucl_df = (
    ucl_raw.groupby("CustomerID")
    .agg(
        recency=("InvoiceDate", lambda x: (anchor - x.max()).days),
        frequency=("InvoiceNo", "nunique"),
        monetary=("LineTotal", "sum"),
        avg_basket=("LineTotal", "mean"),
        n_items=("Quantity", "sum"),
    )
    .reset_index()
)
ucl_df["log_monetary"] = np.log1p(ucl_df["monetary"])
ucl_df["log_frequency"] = np.log1p(ucl_df["frequency"])

TAOBAO_PATH = DATA_DIR / "TaobaoUserBehavior.csv"
user_set = set()
chunks = []
for chunk in pd.read_csv(
    TAOBAO_PATH,
    header=None,
    names=["user_id", "item_id", "category_id", "behavior", "timestamp"],
    chunksize=500_000,
):
    chunk = chunk[chunk["user_id"].isin(user_set) | (len(user_set) < 50000)]
    if len(user_set) < 50000:
        for u in chunk["user_id"].unique():
            if len(user_set) >= 50000:
                break
            user_set.add(u)
    chunk = chunk[chunk["user_id"].isin(user_set)]
    if len(chunk):
        chunks.append(chunk)
    if len(user_set) >= 50000 and sum(len(c) for c in chunks) > 200_000:
        break

taobao = pd.concat(chunks, ignore_index=True)
taobao_df = (
    taobao.groupby("user_id")
    .agg(
        n_events=("behavior", "count"),
        n_pv=("behavior", lambda x: (x == "pv").sum()),
        n_cart=("behavior", lambda x: (x == "cart").sum()),
        n_fav=("behavior", lambda x: (x == "fav").sum()),
        n_buy=("behavior", lambda x: (x == "buy").sum()),
        n_categories=("category_id", "nunique"),
        n_items=("item_id", "nunique"),
    )
    .reset_index()
)
taobao_df["cart_rate"] = taobao_df["n_cart"] / taobao_df["n_events"].clip(lower=1)
taobao_df["converted"] = (taobao_df["n_buy"] > 0).astype(int)

print(f"Olist orders: {len(olist_df):,}")
print(f"UCL customers: {len(ucl_df):,}")
print(f"Taobao users: {len(taobao_df):,}")


# High-value customer target

Label customers in the top quartile of total spend.


In [ ]:
ucl_df = ucl_df.copy()
ucl_df["high_value"] = (ucl_df["monetary"] >= ucl_df["monetary"].quantile(0.75)).astype(int)

feats = ["recency", "log_frequency", "avg_basket", "n_items"]
X = ucl_df[feats].values
y = ucl_df["high_value"].values

print(f"High-value customers: {y.sum():,} ({100 * y.mean():.1f}%)")


# Decision tree classifier


In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

dt = DecisionTreeClassifier(
    max_depth=8, min_samples_leaf=10, random_state=42, class_weight="balanced"
)
dt.fit(X_train, y_train)
dt_train_pred = dt.predict(X_train)
dt_test_pred = dt.predict(X_test)

print(f"Decision tree train F1: {f1_score(y_train, dt_train_pred):.4f}")
print(f"Decision tree test F1: {f1_score(y_test, dt_test_pred):.4f}")
print(f"Decision tree test accuracy: {accuracy_score(y_test, dt_test_pred):.4f}")


A single decision tree can fit training noise when depth is unrestricted. Limiting max_depth and min_samples_leaf reduces overfitting on the UCL customer sample.


# Random forest classifier


In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=200, max_depth=8, min_samples_leaf=10,
    random_state=42, class_weight="balanced",
)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

print(f"Random forest test F1: {f1_score(y_test, rf_pred):.4f}")
print(f"Random forest test accuracy: {accuracy_score(y_test, rf_pred):.4f}")


# Hyperparameter tuning


In [ ]:
from sklearn.model_selection import GridSearchCV

grid = GridSearchCV(
    RandomForestClassifier(random_state=42, class_weight="balanced"),
    {"n_estimators": [100, 200], "max_depth": [6, 8, None], "min_samples_leaf": [5, 10]},
    cv=3,
    scoring="f1",
    n_jobs=-1,
)
grid.fit(X_train, y_train)
best_pred = grid.best_estimator_.predict(X_test)

print(f"Best params: {grid.best_params_}")
print(f"Tuned random forest test F1: {f1_score(y_test, best_pred):.4f}")


# Model comparison and feature importance


In [ ]:
import matplotlib.pyplot as plt

dt_f1 = f1_score(y_test, dt_test_pred)
rf_f1 = f1_score(y_test, rf_pred)
tuned_f1 = f1_score(y_test, best_pred)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].bar(
    ["Decision tree", "Random forest", "Tuned RF"],
    [dt_f1, rf_f1, tuned_f1],
    color=["#E15759", "#4E79A7", "#59A14F"],
)
axes[0].set_ylabel("Test F1")
axes[0].set_title("Week 6: Tree vs forest (UCL high-value customers)")
axes[0].set_ylim(0, 1.0)

imp = grid.best_estimator_.feature_importances_
axes[1].barh(feats, imp, color="darkgreen")
axes[1].set_xlabel("Importance")
axes[1].set_title("Random forest feature importances")

plt.tight_layout()
plt.show()


# Olist review-score random forest

Predict high review scores (≥ 4) from order economics.


In [ ]:
from sklearn.metrics import roc_auc_score

olist_feats = ["total_price", "freight_value", "n_items", "payment_installments", "delivery_days"]
valid = olist_df[olist_feats + ["review_score"]].dropna()
Xo = valid[olist_feats].values
yo = (valid["review_score"] >= 4).astype(int).values

Xo_train, Xo_test, yo_train, yo_test = train_test_split(
    Xo, yo, test_size=0.2, random_state=42, stratify=yo
)
rf_o = RandomForestClassifier(n_estimators=150, max_depth=8, random_state=42)
rf_o.fit(Xo_train, yo_train)
pred_o = rf_o.predict(Xo_test)
proba_o = rf_o.predict_proba(Xo_test)[:, 1]

print(f"Olist review F1: {f1_score(yo_test, pred_o):.4f}")
print(f"Olist review ROC-AUC: {roc_auc_score(yo_test, proba_o):.4f}")
